# Assignment 9

### Imports

In [2]:
import os
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json

### Function for making train/validation/test - split

In [9]:
def split_csvfiles(datafolder, random_seed):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)
    
    random.seed(random_seed)
    random.shuffle(csv_files)

    # Train/Validation/Test with proportions 80/10/10
    train_n = int(len(csv_files) * 0.8)
    val_n = int(len(csv_files) * 0.1)
    test_n = len(csv_files) - train_n - val_n

    # Split
    train_files = csv_files[:train_n]
    val_files = csv_files[train_n: train_n + val_n]
    test_files = csv_files[train_n + test_n:]

    return train_files, val_files, test_files

### Modified enis data load function to not rely on locally defined data_dir within function

In [10]:
def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined

### Apply to get the dataframes

In [11]:
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
random_seed = 42

train_files, val_files, test_files = split_csvfiles(datafolder, random_seed)

train_data = load(train_files, datafolder)
val_data = load(val_files, datafolder)
test_data = load(test_files, datafolder)

### Function for splitting input (x, y) and target (z)

In [19]:
def input_target_split(dataframe):
    input_cols = []
    target_cols = []
    for c in dataframe.columns:
        if c.endswith("_x") or c.endswith("_y"):
            input_cols.append(c)
        if c.endswith("_z"):
            target_cols.append(c)
    
    input_data = dataframe[input_cols]
    target_data = dataframe[target_cols]
    return input_data, target_data

### Apply to get X_train, Y_train e.t.c

In [20]:
x_train, y_train = input_target_split(train_data)
x_val, y_val = input_target_split(val_data)
x_test, y_test = input_target_split(test_data)